# Downsampling approaches
Exploring & implementing the different approaches to downsampling a genome

## MinHashing
Confirmed downsampling approach from Special Course.

Pros:
- Fast & efficient

Cons:
- Hashing algorithm unavailable; backtracking hash values to kmers is not possible (e.g. feature attributions for NNs)

### Constructing singular signatures

In [ ]:
import os, sys 
from manipulations import construct_SM_sketches
from io_operations import presence_matrix
from paths import raw_data_path, data_prod_path

### Phage Minhash Sketch Construction ###
pk = 12
pn = 500
phage_outdir = f"PhageMinhash_n{pn}_k{pk}/"
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                      k = pk, 
                      outdir = phage_outdir, 
                      quiet = False,
                      sourmash_parameters=[pn, 0])

### Bacteria Minhash Sketch Construction ###
bk = 12
bn = 500
bact_outdir = f"BactMinhash_n{bn}_k{bk}/"
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                      k = bk, 
                      outdir = bact_outdir, 
                      quiet = False,
                      sourmash_parameters=[bn, 0])


Created output directory: PhageMinhash_n500_k12/
------- Constructing MinHashes -------


Constructing minhashes for all records: 100%|██████████| 23/23 [00:02<00:00, 11.39seq/s]


------- Saving Sketches -------
------- Process Completed -------
Created output directory: BactMinhash_n500_k12/
------- Constructing MinHashes -------


Constructing minhashes for all records: 100%|██████████| 280/280 [17:12<00:00,  3.69s/seq] 


------- Saving Sketches -------
------- Process Completed -------
500 12 500 12
Loading phage minhash sketches from: /Users/asbjornhansen/PredictPhagePPI/data_prod/PhageMinhash_n500_k12/
Loading bacteria minhash sketches from: /Users/asbjornhansen/PredictPhagePPI/data_prod/BactMinhash_n500_k12/
Error loading minhash sketches: [Errno 2] No such file or directory: '/Users/asbjornhansen/PredictPhagePPI/data_prod/PhageMinhash_n500_k12/'


FileNotFoundError: [Errno 2] No such file or directory: '/Users/asbjornhansen/PredictPhagePPI/data_prod/PresMat_bn500_bk12_pn500_pk12/binary_matrix'

In [8]:
from pickle import dump
### Presence Matrix
presence_outdir = f"SM_sketches/PresMat_bn{bn}_bk{bk}_pn{pn}_pk{pk}/"
try:
    os.makedirs(data_prod_path+presence_outdir)
except FileExistsError: # directory already exists
    pass

try:
    binary_matrix, entity_to_index, minhash_to_index, phage_minhash_data, bact_minhash_data = presence_matrix(
        phage_minhash_dir=data_prod_path+"SM_sketches/"+phage_outdir, 
        bact_minhash_dir=data_prod_path+"SM_sketches/"+bact_outdir,
        k=[bk, pk],
        n=[bn, pn],
        reversecomp_data=False, TS=True)
except Exception as e:
    raise RuntimeError(f"Failed to run presence_matrix succesfully, exception: {e}")

try:
    with open(data_prod_path+presence_outdir+"binary_matrix.pkl", "wb") as binary_matrix_file:
        dump(binary_matrix, binary_matrix_file)
    with open(data_prod_path+presence_outdir+"entity_to_index.pkl", "wb") as entity_to_index_file:
        dump(entity_to_index, entity_to_index_file)
    with open(data_prod_path+presence_outdir+"minhash_to_index.pkl", "wb") as minhash_to_index_file:
        dump(minhash_to_index, minhash_to_index_file)
    with open(data_prod_path+presence_outdir+"phage_minhash_data.pkl", "wb") as phage_minhash_data_file:
        dump(phage_minhash_data, phage_minhash_data_file)
    with open(data_prod_path+presence_outdir+"bact_minhash_data.pkl", "wb") as bact_minhash_data_file:
        dump(bact_minhash_data, bact_minhash_data_file)
except Exception as e:
    print(f"Failed to save presence_matrix results to {presence_outdir}:\n{e}")

500 12 500 12
Loading phage minhash sketches from: /Users/asbjornhansen/PredictPhagePPI/data_prod/SM_sketches/PhageMinhash_n500_k12/
Loading bacteria minhash sketches from: /Users/asbjornhansen/PredictPhagePPI/data_prod/SM_sketches/BactMinhash_n500_k12/

Unique minhashes extracted with len: 21135

Binary presence matrix created with shape: (133, 21135)
Sample rows (entities): ['Abuela', 'Amona', 'Crus', 'Echoes', 'FO3A_23_KMC_reoriented']
Sample columns (minhashes): [43562906906, 652945146928, 2603753039444, 5145884020526, 5403791273531]


### Constructing multiple signatures

In [ ]:
from manipulations import construct_SM_sketches
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

for k in [6, 9, 12, 15, 18, 24]:
    for n in [50, 100, 500, 1000, 5000]:
        #Phages minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                            k = k, 
                            outdir = f"PhageMinhash_n{n}_k{k}_rev/", 
                            quiet = False,
                            sourmash_parameters=[n, 0],
                            include_reverse=True)
        
        #Bacteria minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                        k = k, 
                        outdir = f"BactMinhash_n{n}_k{k}_rev/", 
                        quiet = False,
                        sourmash_parameters=[n, 0],
                        include_reverse=True)

## Novel decompisition method
Develop a new method, where i can backtrack the decomposed integers, to its original kmer sequences

Encoder Process:
1) encode the forward k-mer to bit-level integer
2) encode its reverse complement 
3) keep only the smaller of the two

Decomposition Process:
1) divide genome into kmer of size k
2) keep only every x entry, where x = n/genome_kmer_size (n = sig size)
3) save to disk

### Encoder

In [1]:
from decompositions import KmerCodec
codec = KmerCodec()
my_kmer = "GATCGACT"
k_size = len(my_kmer)

# 1. Decompose to integer
encoded_val = codec.encode_with_revcomp(my_kmer)
print(f"Original: {my_kmer}")
print(f"Integer representation: {encoded_val}") # 8864 in decimal

# 2. Backtrack to sequence
decoded_val = codec.decode(encoded_val, k_size)
print(f"Backtracked: {decoded_val}")

Original: GATCGACT
Integer representation: 36773937
Backtracked: AGTCGATC


### Decomposition

In [2]:
from decompositions import Decompose
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

k = 12
n = 400

# The 'with' block handles the creation and deletion of tmp automatically
with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="phage", sourmash_like=True, 
               custom_dir_name=f"encode4bit_n{n}_k{k}") as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/phage_cleaned.fasta"):
        print(line)

with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="bact", sourmash_like=True,
               custom_dir_name=f"encode4bit_n{n}_k{k}") as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta"):
        print(line)

Using custom directory name: ../data_prod/encoded_sketches/phage_encode4bit_n400_k12
Initialized Decompose with k=12, n=400, entity_type='phage', sourmash_like=True


Processing phage FASTA: 23rec [00:00, 128.17rec/s]


Using custom directory name: ../data_prod/encoded_sketches/bact_encode4bit_n400_k12
Initialized Decompose with k=12, n=400, entity_type='bact', sourmash_like=True


Processing bact FASTA: 280rec [00:52,  5.34rec/s]


# Statistical Eval on down-sampled signatures